# Problem: Implement a Transformer Encoder Block for Text Classification

### Problem Statement
You are tasked with implementing a **Transformer Encoder Block from scratch** for text classification on the **AG News** dataset using PyTorch. Instead of using PyTorch's built-in transformer modules, you must implement the key components manually: Multi-Head Attention (MHA), Feed-Forward Network, Layer Normalization, and Residual Connections.

### Requirements
1. **Implement Custom Components**:
   - Create a custom `MultiHeadAttention` class that implements scaled dot-product attention with multiple heads.
   - Create a custom `FeedForward` class for the position-wise feed-forward network.
   - Create a custom `TransformerEncoderBlock` that combines LayerNorm, MHA, FeedForward, and residual connections.

2. **Define the Classification Model**:
   - Use embedding layers for input tokens.
   - Apply positional encoding to the embeddings.
   - Pass through your custom `TransformerEncoderBlock`.
   - Use a classification head to predict one of 4 classes.

3. **Train on AG News Dataset**:
   - Load the AG News dataset (4 classes: World, Sports, Business, Sci/Tech).
   - Preprocess and tokenize the text data.
   - Train the model and evaluate performance.

### Constraints
- You must not use `nn.TransformerEncoder` or `nn.TransformerEncoderLayer`. Implement from scratch.
- The model should handle variable-length sequences using proper masking.
- Ensure proper implementation of attention mechanism with scaling.

<details>
  <summary>💡 Hint</summary>
  For Multi-Head Attention: Split the input into multiple heads, compute attention for each head independently, then concatenate and project.
  For the Encoder Block: Apply LayerNorm before each sub-layer (Pre-LN architecture), add residual connections after each sub-layer.
  Use positional encoding to inject sequence order information since attention is permutation-invariant.
</details>

## Step 1: Data Acquisition

Load the **AG News** dataset using `torchtext` or `datasets` library. The dataset contains news articles classified into 4 categories.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import math
from collections import Counter
import numpy as np

try:
    from torchtext.datasets import AG_NEWS
    from torchtext.data.utils import get_tokenizer
    print("Using torchtext for AG_NEWS dataset")
    USE_TORCHTEXT = True
except:
    print("torchtext not available, using datasets library")
    from datasets import load_dataset
    USE_TORCHTEXT = False

# Load dataset
if USE_TORCHTEXT:
    train_iter, test_iter = AG_NEWS(split=('train', 'test'))
    train_data = list(train_iter)
    test_data = list(test_iter)
else:
    dataset = load_dataset('ag_news')
    train_data = [(row['label'], row['text']) for row in dataset['train']]
    test_data = [(row['label'], row['text']) for row in dataset['test']]

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Number of classes: 4")
print(f"\nSample: {train_data[0]}")

## Step 2: Data Wrangling

Preprocess the text data:
- Tokenize the text using a simple tokenizer or word-level tokenization
- Build a vocabulary from the training data
- Convert text to sequences of token indices
- Pad sequences to a fixed length
- Create DataLoaders for training and testing

In [ ]:
# Tokenizer
if USE_TORCHTEXT:
    tokenizer = get_tokenizer('basic_english')
else:
    def tokenizer(text):
        return text.lower().split()

# Build vocabulary
def build_vocab(data, tokenizer, min_freq=2):
    counter = Counter()
    for label, text in data:
        counter.update(tokenizer(text))
    
    vocab = {'<pad>': 0, '<unk>': 1}
    idx = 2
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab

vocab = build_vocab(train_data, tokenizer, min_freq=2)
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

# Text to indices
def text_pipeline(text, vocab, tokenizer, max_len=128):
    tokens = tokenizer(text)
    indices = [vocab.get(token, vocab['<unk>']) for token in tokens]
    # Pad or truncate
    if len(indices) < max_len:
        indices += [vocab['<pad>']] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return indices

# Custom Dataset
class TextDataset(Dataset):
    def __init__(self, data, vocab, tokenizer, max_len=128):
        self.data = data
        self.vocab = vocab
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        label, text = self.data[idx]
        indices = text_pipeline(text, self.vocab, self.tokenizer, self.max_len)
        return torch.tensor(indices, dtype=torch.long), torch.tensor(label - 1, dtype=torch.long)

# Create datasets and dataloaders
MAX_LEN = 128
BATCH_SIZE = 32

train_dataset = TextDataset(train_data, vocab, tokenizer, MAX_LEN)
test_dataset = TextDataset(test_data, vocab, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## Step 3: Model Implementation

Implement the following components from scratch:

### 3.1 Multi-Head Attention
- Implement scaled dot-product attention: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k))V
- Split into multiple heads and concatenate results

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Q, K, V: (batch_size, num_heads, seq_len, d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights
    
    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.size()
        
        # Linear projections and reshape for multi-head
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply attention
        attn_output, _ = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        output = self.W_o(attn_output)
        
        return output

### 3.2 Position-wise Feed-Forward Network
- Two linear transformations with ReLU activation: FFN(x) = max(0, xW1 + b1)W2 + b2

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

### 3.3 Positional Encoding
- Add positional information to embeddings using sine and cosine functions

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Add batch dimension
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

### 3.4 Transformer Encoder Block
- Combine all components with Layer Normalization and residual connections

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerEncoderBlock, self).__init__()
        
        # Multi-head attention
        self.attention = MultiHeadAttention(d_model, num_heads)
        
        # Feed-forward network
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Multi-head attention with pre-norm and residual
        attn_output = self.attention(self.norm1(x), mask)
        x = x + self.dropout1(attn_output)
        
        # Feed-forward with pre-norm and residual
        ff_output = self.feed_forward(self.norm2(x))
        x = x + self.dropout2(ff_output)
        
        return x

### 3.5 Complete Classification Model
- Combine embedding, positional encoding, encoder block, and classification head

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_classes, max_len, dropout=0.1):
        super(TransformerClassifier, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        # Transformer encoder block
        self.encoder_block = TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
        
        # Classification head
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, num_classes)
    
    def forward(self, x, mask=None):
        # x: (batch_size, seq_len)
        
        # Embedding and positional encoding
        x = self.embedding(x)  # (batch_size, seq_len, d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        
        # Pass through encoder block
        x = self.encoder_block(x, mask)
        
        # Pool by taking mean across sequence length
        x = torch.mean(x, dim=1)  # (batch_size, d_model)
        
        # Classification
        x = self.fc(x)  # (batch_size, num_classes)
        
        return x

## Step 4: Training

Train the model using:
- Loss Function: CrossEntropyLoss
- Optimizer: Adam with learning rate scheduling
- Training loop with gradient clipping

In [ ]:
# Hyperparameters
D_MODEL = 128
NUM_HEADS = 8
D_FF = 512
NUM_CLASSES = 4
DROPOUT = 0.1
LEARNING_RATE = 0.0001
NUM_EPOCHS = 5
MAX_GRAD_NORM = 1.0

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TransformerClassifier(
    vocab_size=vocab_size,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_classes=NUM_CLASSES,
    max_len=MAX_LEN,
    dropout=DROPOUT
).to(device)

print(f"Model initialized with {sum(p.numel() for p in model.parameters())} parameters")
print(f"Training on device: {device}")

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training loop
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (texts, labels) in enumerate(train_loader):
        texts, labels = texts.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(texts)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        if (batch_idx + 1) % 500 == 0:
            print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Step [{batch_idx+1}/{len(train_loader)}], "
                  f"Loss: {loss.item():.4f}, Acc: {100*correct/total:.2f}%")
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] Summary - Avg Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%\n")

## Step 5: Evaluation

Evaluate the model using:
- Accuracy on the test set
- Optional: Precision, Recall, F1-Score per class

In [ ]:
# Evaluation function
def evaluate(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for texts, labels in data_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    return accuracy

# Evaluate on test set
test_accuracy = evaluate(model, test_loader, device)
print(f"Test Accuracy: {test_accuracy:.2f}%")

# Per-class evaluation (optional)
def evaluate_per_class(model, data_loader, device, num_classes=4):
    model.eval()
    class_correct = [0] * num_classes
    class_total = [0] * num_classes
    
    with torch.no_grad():
        for texts, labels in data_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            _, predicted = torch.max(outputs, 1)
            
            for i in range(labels.size(0)):
                label = labels[i].item()
                class_total[label] += 1
                if predicted[i] == labels[i]:
                    class_correct[label] += 1
    
    class_names = ['World', 'Sports', 'Business', 'Sci/Tech']
    print("\nPer-class Accuracy:")
    for i in range(num_classes):
        accuracy = 100 * class_correct[i] / class_total[i] if class_total[i] > 0 else 0
        print(f"{class_names[i]}: {accuracy:.2f}% ({class_correct[i]}/{class_total[i]})")

evaluate_per_class(model, test_loader, device)